In [1]:
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, FewShotPromptTemplate, PromptTemplate
from langchain_community.document_loaders import PyPDFLoader, TextLoader 
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [3]:
load_dotenv()

True

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage,SystemMessage,BaseMessage


# Step 1: Create a template using input_variables
prompt = ChatPromptTemplate(
    # input_variables=["context", "question"],  # use 'question' instead of 'messages' to avoid conflict
    messages=[
        SystemMessage(content="You act like virtual assistant of Munna. Answer the question based on the given context. If you don't have the context, just say that you don't know, don't try to make up an answer."),
        HumanMessage(content="Context: {context} \n\nQuestion: {question}" ),
    ]
)

# Step 2: Provide values
formatted_messages = prompt.format_messages(
    context="Munna is a data scientist working in fintech.",
    question="What does Munna do?"
)

In [89]:
prompt.invoke(
    {
        "context": "Munna is a data scientist working in fintech.",
        "question": "What does Munna do?"
    }
)

ChatPromptValue(messages=[SystemMessage(content="You act like virtual assistant of Munna. Answer the question based on the given context. If you don't have the context, just say that you don't know, don't try to make up an answer.", additional_kwargs={}, response_metadata={}), HumanMessage(content='Context: {context}\n\nQuestion: {question}', additional_kwargs={}, response_metadata={})])

In [88]:
formatted_messages

[SystemMessage(content="You act like virtual assistant of Munna. Answer the question based on the given context. If you don't have the context, just say that you don't know, don't try to make up an answer.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Context: {context}\n\nQuestion: {question}', additional_kwargs={}, response_metadata={})]

In [4]:
## Embedding
doccument = PyPDFLoader('Munna Biography.pdf').load()
embedding = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=500)
splitter = RecursiveCharacterTextSplitter(chunk_size=600, separators=["Chapter", "\n\n", "\n", " ", ""])
splitted_doccuments = splitter.split_documents(doccument)
vectorstore = FAISS.from_documents(splitted_doccuments, embedding)
retriever = vectorstore.as_retriever(search_kwargs={"k":5,},search_type="mmr")

In [5]:
retriever.invoke("Education background of Munna")

[Document(metadata={'source': 'Munna Biography.pdf', 'page': 2}, page_content='with ambition, and passion with purpose.\nMahmud Hasan Munna’s story is a testament to curiosity, resilience, and the power of self-directed learning.\nIt is a journey of a quiet mind that can create profound impact — a journey that is only just beginning.\n3'),
 Document(metadata={'source': 'Munna Biography.pdf', 'page': 1}, page_content='Chapter 4: Growing Through Responsibility\nI moved from Junior Data Scientist to Data Scientist at SSL Wireless within a year and a half. Suddenly, I was\nleading teams, making technical decisions, and deploying models in production. My growth accelerated not\nbecause the tasks were easy, but because the challenges forced me to expand my capabilities.\nLater , I joined Wegro Technologies Limited, first as a Data Scientist, and then as a Senior Data Scientist.\nHere, I learned the intersection of technology and business. I realized that solving a problem technically'),
 Doc

In [6]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    context: List[str]

In [ ]:
def ragnode(state: ChatState) -> ChatState:
    query = state["messages"][-1].content
    docs = retriever.invoke(query)

    return {
        "messages": state["messages"],
        "context": [doc.page_content for doc in docs]
    }

In [69]:
contexts = ragnode({
    "messages": [HumanMessage(content="Whats the education background of Munna?")],
    "context": []
})

In [71]:
contexts

{'messages': [HumanMessage(content='Whats the education background of Munna?', additional_kwargs={}, response_metadata={})],
 'context': ['with ambition, and passion with purpose.\nMahmud Hasan Munna’s story is a testament to curiosity, resilience, and the power of self-directed learning.\nIt is a journey of a quiet mind that can create profound impact — a journey that is only just beginning.\n3',
  'Chapter 4: Growing Through Responsibility\nI moved from Junior Data Scientist to Data Scientist at SSL Wireless within a year and a half. Suddenly, I was\nleading teams, making technical decisions, and deploying models in production. My growth accelerated not\nbecause the tasks were easy, but because the challenges forced me to expand my capabilities.\nLater , I joined Wegro Technologies Limited, first as a Data Scientist, and then as a Senior Data Scientist.\nHere, I learned the intersection of technology and business. I realized that solving a problem technically',
  'Life is not all cod

In [76]:
print("\n\n".join(contexts["context"]))

with ambition, and passion with purpose.
Mahmud Hasan Munna’s story is a testament to curiosity, resilience, and the power of self-directed learning.
It is a journey of a quiet mind that can create profound impact — a journey that is only just beginning.
3

Chapter 4: Growing Through Responsibility
I moved from Junior Data Scientist to Data Scientist at SSL Wireless within a year and a half. Suddenly, I was
leading teams, making technical decisions, and deploying models in production. My growth accelerated not
because the tasks were easy, but because the challenges forced me to expand my capabilities.
Later , I joined Wegro Technologies Limited, first as a Data Scientist, and then as a Senior Data Scientist.
Here, I learned the intersection of technology and business. I realized that solving a problem technically

Life is not all code, models, and data pipelines. I find joy in playing guitar , singing, watching movies, and
cricket. Cricket, in particular , resonates deeply — scoring ru

In [63]:
def chatnode(state: ChatState) -> ChatState:
    promt = ChatPromptTemplate(
        messages=[
            SystemMessage(content="You act like vertual assistant of Munna. Answer the question based on the given context. If you don't have the context, just say that you don't know, don't try to make up an answer."),
            HumanMessage(content="Context: {context}\n\nQuestion: {messages}"),
        ]
    )
    formatted_messages = promt.format_messages(
        context="\n\n".join(state["context"]),
        messages=state["messages"]
    )
    
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)
    response = llm.invoke(formatted_messages)
    
    
    return {"messages": [response] }


In [ ]:
# initial_state = {
#     "messages": [HumanMessage(content="First Job")],
#     "context": []
# }

In [62]:
# ragnode(initial_state)

In [47]:
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)
# response = llm.invoke(promt.format_prompt(messages=state["messages"][-1].content, context="\n\n".join(state["context"])))

In [24]:
memory = MemorySaver()

In [64]:
workflow = StateGraph(ChatState)
workflow.add_node('rag', ragnode)
workflow.add_node('chat', chatnode)

workflow.add_edge(START, 'rag')
workflow.add_edge('rag', 'chat')
# workflow.add_edge('chat', END)
chatbot = workflow.compile(checkpointer=memory)
chatbot



In [67]:
result = chatbot.invoke(
    {"messages": [HumanMessage(content="Can you tell me about Munna's First Job?")]}
     , config={"configurable": {"thread_id": "90012"}},
        
)

In [68]:
result['messages']

[HumanMessage(content="Can you tell me about Munna's First Job?", additional_kwargs={}, response_metadata={}, id='952f95e4-8c7f-4b6f-a07a-ef71693f7c76'),
 AIMessage(content="I'm sorry, but I don't have access to the context of the conversation. If you provide me with the context, I'll be happy to help answer your question.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 65, 'total_tokens': 100, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-c94375a5-6a03-4d85-9075-2d8e4c45c0d8-0', usage_metadata={'input_tokens': 65, 'output_tokens': 35, 'total_tokens': 100})]

In [10]:
from IPython.display import display
display(chatbot)


In [11]:
llm = ChatOpenAI(model='gpt-3.5-turbo')

In [4]:
## Steps
## Load Document, Embedding, Vector Store, Retrieval QA, LLM Chain

In [5]:
doccument = PyPDFLoader('Munna Biography.pdf').load()

In [6]:
embedding = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=500)

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600, separators=["Chapter", "\n\n", "\n", " ", ""])

In [8]:
splitted_doccuments = splitter.split_documents(doccument)

In [9]:
vectorstore = FAISS.from_documents(splitted_doccuments, embedding)

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k":5,},search_type="similarity")

In [11]:
retriever.invoke("What is Munna's First Job?")

[Document(metadata={'source': 'Munna Biography.pdf', 'page': 0}, page_content='Chapter 3: First Steps in the Professional World\nMy first brush with earning independently came through tutoring students. Teaching sharpened my ability\nto break down complex ideas, explain concepts clearly, and cultivate patience — lessons that would later\nbecome invaluable in my professional journey.\nOn 14 March 2022, I officially stepped into the world of data science as a Junior Data Scientist at SSL\nWireless, one of Bangladesh’s largest fintech companies. It was a mixture of excitement and anxiety. I was'),
 Document(metadata={'source': 'Munna Biography.pdf', 'page': 2}, page_content='with ambition, and passion with purpose.\nMahmud Hasan Munna’s story is a testament to curiosity, resilience, and the power of self-directed learning.\nIt is a journey of a quiet mind that can create profound impact — a journey that is only just beginning.\n3'),
 Document(metadata={'source': 'Munna Biography.pdf', 'pa

In [12]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
)

In [13]:
compression_retriver =  ContextualCompressionRetriever(
    base_compressor=LLMChainExtractor.from_llm(llm=llm),
    base_retriever=retriever
)

In [15]:
compression_retriver.invoke("Munna's First Job?")

[]

In [16]:
print(splitted_doccuments[1].page_content)

Chapter 1: Roots of Curiosity
I was born on 26 June 1998, in the small town of Mohanpur Upazila, Rajshahi, Bangladesh. My childhood
was filled with freedom, curiosity, and exploration. I was a child who wanted to understand how things
worked — not just follow instructions. I dismantled objects, tinkered with ideas, and often found myself lost
in thought, imagining possibilities.
My father , quietly strong and disciplined, influenced me more than anyone else. From him, I learned the
value of integrity, patience, and thoughtful action. My family was supportive, adventurous, and encouraging


In [17]:
chat_template = PromptTemplate(
    template="""Act like you are Munna. Answer the user query how Munna would, following the context. 
If you do not have sufficient context, say "I cannot answer that."

Context:
{context}

Question: {question}""",
    input_variables=['context', 'question']
)

In [18]:
llm = ChatOpenAI(model='gpt-5.1')


In [19]:
# chain = retriever| chat_template | llm

In [22]:
question = "list all the achievements of Munna?"
retrived_info = retriever.invoke(question)
context_text = "\n\n".join(doc.page_content for doc in retrived_info)
final_promt = chat_template.format(context=context_text, question=question)
print(llm.invoke(final_promt).content)

Here are the achievements mentioned about me in the provided context:

1. **Professional Role & Leadership**
   - Became a **Senior Data Scientist** leading AI initiatives, building systems, and mentoring teams.

2. **Major Data & AI Contributions**
   - **Designed a 20 million–customer profiling system**  
     - Became the backbone of targeted marketing.  
     - Generated **over 1 crore BDT in revenue**.
   - **Developed an eKYC system**  
     - Saved **500,000 BDT**.
   - **Built a company-wide data warehouse**  
     - Transformed the **speed and efficiency of decision-making**.

3. **Early Professional Experience**
   - Started career as a **Junior Data Scientist at SSL Wireless** (one of Bangladesh’s largest fintech companies) on **14 March 2022**.
   - **Tutored students** early on, which developed:
     - Skill in breaking down complex ideas.
     - Clear explanation ability.
     - Patience and teaching capability.

4. **Personal & Philosophical Achievements**
   - Built a c